# Feature Engineering Study - Tabular (Regression)

Comparative study of feature engineering techniques for regression on the
**California Housing** dataset (20.640 samples, 8 features, target = median price
in $100.000). Each technique is evaluated with 3 models: **LinearRegression**
(scale-sensitive), **LightGBM** (gradient boosting) and **RandomForest**.

Central question: *how much does feature engineering help each type of model?*

In [1]:
import numpy as npimport pandas as pdimport timeimport warningswarnings.filterwarnings('ignore')from sklearn.datasets import fetch_california_housingfrom sklearn.model_selection import train_test_splitfrom sklearn.preprocessing import StandardScaler, MinMaxScaler, PolynomialFeatures, KBinsDiscretizerfrom sklearn.decomposition import PCAfrom sklearn.linear_model import LinearRegressionfrom sklearn.ensemble import RandomForestRegressorfrom sklearn.metrics import mean_absolute_error, r2_scoreimport lightgbm as lgbSEED = 42np.random.seed(SEED)print('Imports OK')

Imports OK


## 1. Dataset Loading

In [2]:
data = fetch_california_housing()X = pd.DataFrame(data.data, columns=data.feature_names)y = data.targetprint(f'Shape: {X.shape}')X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)print(f'Train: {len(X_train)}, Test: {len(X_test)}')

Shape: (20640, 8)
Train: 16512, Test: 4128


## 2. Feature Engineering Techniques

| # | Technique | Description |
|---|---------|------------|
| 1 | Raw | Original features |
| 2 | Standardized | StandardScaler |
| 3 | MinMax | MinMaxScaler |
| 4 | Polynomial (d=2) | 44 features |
| 5 | Poly interactions | 36 features (no x^2) |
| 6 | Log transform | log1p on skewed features |
| 7 | Binning | Discretization into 10 bins |
| 8 | PCA (95%) | Reduction to 6 features |
| 9 | Geo features | distance to the center + ratios - 12 features |
| 10 | Combined | Log + Geo + Poly + Standard - 55 features |

In [3]:
# Feature engineering functions (see _run_fe_tabular.py for the full code)# Each function receives X_train, X_test and returns the transformed matrices.TECHNIQUES = [    ('1. Raw', fe_raw),    ('2. Standardized', fe_standardized),    ('3. MinMax', fe_minmax),    ('4. Polynomial (d=2)', fe_polynomial),    ('5. Poly interactions', fe_poly_interactions),    ('6. Log transform', fe_log_transform),    ('7. Binning (10 bins)', fe_binning),    ('8. PCA (95% var)', fe_pca),    ('9. Geo features', fe_geo),    ('10. Combined', fe_combined),]

## 3. Results

In [4]:
# Results of the full run (script _run_fe_tabular.py)# Each technique x 3 models = 30 experiments

Technique                           Model              R2       MAE      Features   Time
------------------------------------------------------------------------------------------
1. Raw                              LinearRegression   0.5758   0.5332   8          0.00
1. Raw                              LightGBM           0.8360   0.3078   8          0.33
1. Raw                              RandomForest       0.8051   0.3275   8          7.36
2. Standardized                     LinearRegression   0.5758   0.5332   8          0.03
2. Standardized                     LightGBM           0.8386   0.3055   8          3.57
2. Standardized                     RandomForest       0.8053   0.3274   8          16.20
3. MinMax                           LinearRegression   0.5758   0.5332   8          0.04
3. MinMax                           LightGBM           0.8365   0.3079   8          1.54
3. MinMax                           RandomForest       0.8044   0.3275   8          12.21
4. Polynomial (d=

## 4. Summary Table (R^2 by model)

| Technique | LinearRegression | LightGBM | RandomForest |
|---------|------------------|----------|-------------|
| 1. Raw | 0,5758 | 0,8360 | 0,8051 |
| 2. Standardized | 0,5758 | 0,8386 | 0,8053 |
| 3. MinMax | 0,5758 | 0,8365 | 0,8044 |
| **4. Polynomial (d=2)** | **0,6457** | 0,8346 | 0,7968 |
| 5. Poly interactions | 0,6225 | 0,8339 | 0,7969 |
| 6. Log transform | 0,6114 | 0,8360 | 0,8053 |
| 7. Binning | 0,5858 | 0,8288 | **0,8154** |
| 8. PCA (95%) | 0,4877 | 0,6583 | 0,6422 |
| **9. Geo features** | 0,5945 | **0,8418** | **0,8205** |
| **10. Combined** | **0,7112** | 0,8375 | 0,8045 |

## 5. Analysis

### 1. LinearRegression - Feature Engineering Is Crucial (+13,5 pp)

Linear models benefit strongly: combined (log + poly + geo + standard)
reaches R^2 = 0,7112 - a gain of **+13,5 pp** over the baseline (0,5758). Polynomial
features alone already give +7,0 pp, allowing nonlinear relationships
(e.g.: MedInc x Latitude) to be captured without switching models. Log transform also helps (+3,6 pp)
by reducing the impact of outliers in features with an asymmetric distribution
(AveRooms, Population, AveOccup).

### 2. LightGBM - Little Benefit (+0,6 pp)

Gradient boosting trees are invariant to scale and capture nonlinearities
natively via splits. The only technique that helps is **Geo features** (+0,6 pp),
because it creates domain-knowledge features that the trees cannot infer
automatically (e.g.: the euclidean distance to the center is not a function of
univariate splits). All the other techniques (scaling, polynomial, PCA) bring
no relevant gain.

### 3. RandomForest - Intermediate Benefit (+1,5 pp)

RandomForest benefits from **Geo features** (+1,5 pp) for the same reason as
LightGBM. **Binning** also helps (+1,0 pp): discretization into 10 bins facilitates
the splits in the trees, which have limited depth and cannot capture
fine boundaries in continuous features.

### 4. PCA - Harmful for All Models

PCA lost 8,8 to 17,8 pp on every model. The problem is that PCA reduces the
dimensionality by projecting onto components that maximize variance - but
variance is not a synonym for predictive power. In this dataset, Latitude and Longitude
are collinear, and PCA merges them into a component that loses the directional
information (north-south) crucial for real-estate prices.

### Conclusions

1. **The value of feature engineering depends on the model**: LinearRegression gained
   13,5 pp; LightGBM only 0,6 pp. The effort should be proportional to the sensitivity
   of the model.

2. **Polynomial features are the most impactful technique for linear models**
   (+7,0 pp alone), allowing nonlinearities to be captured without switching models.

3. **Trees only benefit from domain-knowledge features** (Geo features),
   not from mathematical transforms (scaling, PCA).

4. **PCA should be avoided** - here it lost 9-18 pp on every model.

5. **Recommendation**: for linear models, Combined (log + poly + geo + standard)
   is excellent. For trees, Raw + Geo features is already sufficient.